In [11]:
import mlflow
# import mlflow.sklearn
import optuna

from optuna.integration.mlflow import MLflowCallback
from optuna.pruners import MedianPruner

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    OneHotEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

import json

In [12]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

In [13]:
# =========================================================
# Columns
# =========================================================

nomod_columns = [
    'HasCrCard',
    'IsActiveMember',
]

dummyfy_columns = [
    'Card Type',
    'NumOfProducts',
    'Geography',
    'Gender'
]

norm_std_columns = [
    'Balance',
    'Point Earned',
    'CreditScore',
    'Age',
    'Tenure',
    'Satisfaction Score',
    'EstimatedSalary'
]

In [14]:
# =========================================================
# Config
# =========================================================

RANDOM_STATE = 42
N_SPLITS = 5

EXPERIMENT_NAME = "customer-churn-optuna"

engineer features.
find shap_values to determine most important features

set optuna and mlflow.

In [15]:
# =========================================================
# Build Pipeline
# =========================================================

def build_pipeline(trial):

    # -----------------------------
    # Preprocessing
    # -----------------------------

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat',
             OneHotEncoder(handle_unknown='ignore', drop='first'),
             dummyfy_columns),

            ('num',
             StandardScaler(),
             norm_std_columns),

            ('pass',
             'passthrough',
             nomod_columns)
        ],
        remainder='drop'
    )

    # -----------------------------
    # Random Forest Model
    # -----------------------------

    model = RandomForestClassifier(
        n_estimators=trial.suggest_int(
            'rf_n_estimators',
            100,
            500
        ),
        max_depth=trial.suggest_int(
            'rf_max_depth',
            2,
            20
        ),
        min_samples_split=trial.suggest_int(
            'rf_min_samples_split',
            2,
            20
        ),
        min_samples_leaf=trial.suggest_int(
            'rf_min_samples_leaf',
            1,
            10
        ),
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    # -----------------------------
    # Full Pipeline
    # -----------------------------

    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    return pipeline

In [16]:
# =========================================================
# Objective Function
# =========================================================

def objective(trial):

    pipeline = build_pipeline(trial)

    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        scoring='average_precision',
        cv=cv,
        n_jobs=-1
    )

    mean_auc = np.mean(scores)

    # --------------------------------
    # Report intermediate value
    # (for pruning)
    # --------------------------------

    trial.report(mean_auc, step=0)

    if trial.should_prune():
        raise optuna.TrialPruned()

    return mean_auc

In [17]:
# =========================================================
# MLflow Setup
# =========================================================

import os

ROOT_DIR = os.path.abspath("../")  # or your project root explicitly
MLRUNS_DIR = os.path.join(ROOT_DIR, "mlruns")

mlflow.set_tracking_uri(f"file:{MLRUNS_DIR}")
mlflow.set_experiment(EXPERIMENT_NAME)

mlflc = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="average_precision",
    create_experiment=False,
    mlflow_kwargs={"nested": True}
)

# =========================================================
# Study
# =========================================================

study = optuna.create_study(
    direction='maximize',
    pruner=MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=0
    )
)

/var/folders/bv/50x24wc545x5mclk_t88ryrc0000gn/T/ipykernel_81161/4207547355.py:13: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflc = MLflowCallback(
[I 2026-05-21 18:08:38,017] A new study created in memory with name: no-name-9a1ce872-f529-4b14-8559-903ba129b114


In [18]:
def get_feature_names(preprocessor):
    # Works only after fitting
    feature_names = preprocessor.get_feature_names_out()

    # sklearn returns names like:
    # "cat__Geography_France", "num__Balance", etc.
    return feature_names

In [19]:
# =========================================================
# Run Optimization
# =========================================================

with mlflow.start_run(run_name="optuna_search"):

    study.optimize(
        objective,
        n_trials=250,
        callbacks=[mlflc]
    )

    # --------------------------------
    # Best Metrics
    # --------------------------------

    mlflow.log_metric(
        "best_auc",
        study.best_value
    )

    # --------------------------------
    # Best Params
    # --------------------------------

    mlflow.log_params(study.best_params)

    # --------------------------------
    # Train Best Pipeline
    # --------------------------------

    best_pipeline = build_pipeline(study.best_trial)

    best_pipeline.fit(X_train, y_train)

    preprocessor = best_pipeline.named_steps['preprocessing']

    # --------------------------------
    # log feature names
    # --------------------------------
    # BEFORE columns
    input_features = dummyfy_columns + norm_std_columns + nomod_columns

    # AFTER columns
    output_features = get_feature_names(preprocessor)

    mlflow.log_param("n_input_features", len(input_features))
    mlflow.log_param("n_output_features", len(output_features))
    mlflow.log_dict({
        "input_features": input_features,
        "output_features": output_features
    }, "feature_schema.json")


    # --------------------------------
    # log Metrics
    # --------------------------------
    y_pred = best_pipeline.predict(X_test)
    y_proba = best_pipeline.predict_proba(X_test)[:, 1]

    mlflow.log_metrics({
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_auc": roc_auc_score(y_test, y_proba),
        "train_accuracy": accuracy_score(y_test, y_pred),
        "train_precision": precision_score(y_test, y_pred),
        "train_recall": recall_score(y_test, y_pred),
        "train_f1": f1_score(y_test, y_pred),
        "rain_auc": roc_auc_score(y_test, y_proba),
    })

    mlflow.log_dict(
        classification_report(y_test, y_pred, output_dict=True),
        "classification_report.json"
    )

    mlflow.log_dict(
        confusion_matrix(y_test, y_pred).tolist(),
        "confusion_matrix.json"
    )

    # --------------------------------
    # Save Best Model
    # --------------------------------

    mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        name="best_model",
        #artifact_path="best_model", # artifact_path deprecated soon
        serialization_format="skops"
    )

[I 2026-05-21 18:08:38,555] Trial 0 finished with value: 0.6759558786875954 and parameters: {'rf_n_estimators': 283, 'rf_max_depth': 6, 'rf_min_samples_split': 20, 'rf_min_samples_leaf': 6}. Best is trial 0 with value: 0.6759558786875954.
[I 2026-05-21 18:08:39,088] Trial 1 finished with value: 0.6770706423453292 and parameters: {'rf_n_estimators': 204, 'rf_max_depth': 20, 'rf_min_samples_split': 9, 'rf_min_samples_leaf': 3}. Best is trial 1 with value: 0.6770706423453292.
[I 2026-05-21 18:08:39,805] Trial 2 finished with value: 0.6792850388778242 and parameters: {'rf_n_estimators': 309, 'rf_max_depth': 14, 'rf_min_samples_split': 7, 'rf_min_samples_leaf': 6}. Best is trial 2 with value: 0.6792850388778242.
[I 2026-05-21 18:08:40,751] Trial 3 finished with value: 0.6764897947376384 and parameters: {'rf_n_estimators': 377, 'rf_max_depth': 19, 'rf_min_samples_split': 3, 'rf_min_samples_leaf': 2}. Best is trial 2 with value: 0.6792850388778242.
[I 2026-05-21 18:08:41,186] Trial 4 finished

In [20]:
# =========================================================
# Results
# =========================================================

print("\nBest average_precision_score:")
print(study.best_value)

print("\nBest Params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")


Best average_precision_score:
0.6882736096595738

Best Params:
rf_n_estimators: 414
rf_max_depth: 9
rf_min_samples_split: 20
rf_min_samples_leaf: 1


MLflow Experiment
    └── Optuna Study
            └── Trial Loop
                    ├── Choose hyperparameters (Bayesian optimization)
                    ├── Build preprocessing + model pipeline
                    ├── Run Cross Validation
                    ├── Compute ROC-AUC
                    ├── Report metric to Optuna
                    ├── Optuna decides:
                    │       ├── continue
                    │       └── prune trial
                    └── MLflow logs params + metrics
            └── Best trial selected
    └── Train best model on full training data
    └── Save model to MLflow